# How to Work with Non-Tabular Data

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/non_tabular_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Fnon_tabular_data.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/non_tabular_data.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


QuPrep works with time series (sliding window), sparse matrices, and multi-label targets in addition to standard 2D feature arrays.

In [ ]:
!pip install -q quprep scipy

In [ ]:
import tempfile
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp

import quprep as qd
from quprep import QuPrepWarning

print(f"quprep {qd.__version__}")
rng = np.random.default_rng(0)

## 1. Time series — sliding window

Convert overlapping windows of raw time-step rows into a 2D feature matrix before handing to `NumpyIngester`.

In [ ]:
rng_ts = np.random.default_rng(7)
raw_ts = rng_ts.uniform(0, 1, (20, 2))  # 20 time steps, 2 sensors

window_size = 4
windows = np.array(
    [raw_ts[i : i + window_size].flatten() for i in range(len(raw_ts) - window_size + 1)]
)
ds = qd.NumpyIngester().load(windows)
print(f"Raw time steps   : {len(raw_ts)}")
print(f"Window size      : {window_size}")
print(f"Dataset shape    : {ds.data.shape}  (17 windows × 8 features)")

## 2. Sparse matrices

In [ ]:
sparse_X = sp.random(30, 10, density=0.2, format="csr", random_state=42)
ds_sparse = qd.NumpyIngester().load(sparse_X)
n_elements = sparse_X.shape[0] * sparse_X.shape[1]
print(f"Sparse shape    : {sparse_X.shape}  density={sparse_X.nnz / n_elements:.2f}")
print(f"Dataset shape   : {ds_sparse.data.shape}")

## 3. Multi-label targets

In [ ]:
X_ml = rng.uniform(0, 1, (20, 4))
y_ml = np.column_stack([
    (X_ml[:, 0] > 0.5).astype(int),
    (X_ml[:, 1] > 0.5).astype(int),
])

ds_ml = qd.NumpyIngester().load(X_ml, y=y_ml)
print(f"Feature shape  : {ds_ml.data.shape}")
print(f"Label shape    : {ds_ml.labels.shape}  (2 label columns)")

## 4. Multi-label from CSV

`CSVIngester` loads all columns as features. Use pandas to split features and label columns before passing to `NumpyIngester`.

In [ ]:
ml_csv = "f0,f1,f2,tag_A,tag_B\n"
for i in range(10):
    ml_csv += f"{rng.uniform():.3f},{rng.uniform():.3f},{rng.uniform():.3f},"
    ml_csv += f"{int(rng.uniform()>0.5)},{int(rng.uniform()>0.5)}\n"

with tempfile.NamedTemporaryFile(suffix=".csv", mode="w", delete=False) as f:
    f.write(ml_csv)
    ml_path = f.name

ml_df = pd.read_csv(ml_path)
X_ml_csv = ml_df[["f0", "f1", "f2"]].values
y_ml_csv = ml_df[["tag_A", "tag_B"]].values

ds_ml_csv = qd.NumpyIngester().load(X_ml_csv, y=y_ml_csv)
print(f"Feature shape  : {ds_ml_csv.data.shape}")
print(f"Label shape    : {ds_ml_csv.labels.shape}")

## 5. Encode time series windows

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    result = qd.Pipeline(
        normalizer=qd.Scaler(strategy="minmax_pi"),
        encoder=qd.AngleEncoder(),
    ).fit_transform(ds)

print(f"Circuits  : {len(result.encoded)}")
print(f"Qubits    : {result.encoded[0].metadata['n_qubits']}")
print("(8 features → 8 qubits per window)")